In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [10]:
# load 3 datasets and concatenate them
df1 = pd.read_csv('../data/general/news.csv')
df1['label'] = df1['label'].map({'REAL': 0, 'FAKE': 1}) # map labels to 0 and 1
df2 = pd.read_csv('../data/general/True.csv')
df2['label'] = 0 # all entries in this dataset are real news, so label is 0
df3 = pd.read_csv('../data/general/Fake.csv')
df3['label'] = 1 # all entries in this dataset are fake news, so label is 1
df = pd.concat([df1, df2, df3], ignore_index=True)
df.sample(n=5)

,index,title,text,label,subject,date
21512,NaN,Kremlin says prospects for Putin-Trump meeting...,"DANANG, Vietnam (Reuters) - The prospects for ...",0,worldnews,"November 10, 2017"
26825,NaN,Merkel optimistic EU dispute over refugee dist...,BERLIN (Reuters) - German Chancellor Angela Me...,0,worldnews,"September 10, 2017"
4839,7813.0,Email Reveals What Progressive Think Tank Gain...,"When a prominent, progressive establishment th...",1,NaN,NaN
36703,NaN,Fox Host Claims Obama Used A ‘Raw Onion’ To F...,Anyone who watched Barack Obama break into tea...,1,News,"January 5, 2016"
22946,NaN,French Catalans offer Puigdemont luxury safe-h...,PARIS (Reuters) - French Catalans have a villa...,0,worldnews,"October 24, 2017"


In [11]:
# drop columns that are not needed for classification
df = df.drop(columns=['index', 'title', 'subject', 'date'])

# remove entries where the text length is <200, as they are likely to be noise
df['text_length'] = df['text'].apply(len)
df = df[df['text_length'] >= 200].drop(columns=['text_length'])

In [12]:
# since we are partially using ISOT dataset, we need to perform some minor preprocessing to remove 
# the source of the news from the text, as it can be a strong signal for classification and can lead to overfitting. 
# We will remove any URLs and email addresses from the text.

def clean_text(text):
    # 1. Remove Publisher tags (ISOT specific)
    text = re.sub(r'^.*? - ', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 3. Standard cleaning
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s!\?]', '', text) # Keep text, spaces, !, and ?
    return text

df['text'] = df['text'].apply(clean_text)

In [13]:
# split data into train and test sets
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
# preprocessing function to be used in the pipeline, which will clean the text using the same function as above
preprocess = FunctionTransformer(lambda x: pd.Series(x).apply(clean_text))

# create a pipeline with TfidfVectorizer and SGDClassifier, with hyperparameter weights already tuned previously using GridSearchCV
pipeline = make_pipeline(
    preprocess,
    TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english'),
    SGDClassifier(loss='hinge', penalty=None, alpha=1e-4, random_state=42, learning_rate='pa1', eta0=1.0)
)

In [15]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.969859955348082
F1 Score: 0.9704683305160585
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      4842
           1       0.97      0.97      0.97      5012

    accuracy                           0.97      9854
   macro avg       0.97      0.97      0.97      9854
weighted avg       0.97      0.97      0.97      9854

Confusion Matrix:
 [[4677  165]
 [ 132 4880]]


In [24]:
# The example article is from the BBC, and is a real news article. The model should ideally predict it as 0 (real news).
# Link: https://www.bbc.com/news/articles/c99jxryd9xko
with open('real_article.txt', 'r') as f:
    real_article = f.read()

In [25]:
pipeline.predict([real_article])

array([0])

Since the model predicted that the article was real, then it matches what we know about reality.

What if we try it on a fake news article?

In [26]:
# The example article is from The Gateway Pundit, and is a fake news article. The model should ideally predict it as 1 (fake news).
# I am not giving you the link to this garbage...
with open('fake_article.txt', 'r') as f:
    fake_article = f.read()

In [27]:
pipeline.predict([fake_article])

array([1])

The model marks it as fake!

In [30]:
# predicted probabilities of each article
print("Real article predicted probability of being fake news:", pipeline.decision_function([real_article]))
print("Fake article predicted probability of being fake news:", pipeline.decision_function([fake_article]))

Real article predicted probability of being fake news: [-4.07347168]
Fake article predicted probability of being fake news: [1.04684118]


It seems like the model was extremely certain that the BBC article was real, while it was lightly certain that the GatewayPundit article was actually fake. So potentially, if a fake news article was even better written than the fake article, then it could be passed off as real by the model!

In [ ]:
# 20 largest factors that can predict whether an article is fake or real news, according to the model. 
# We will use the coefficients of the SGDClassifier to find the most important features.

feature_names = pipeline.named_steps['tfidfvectorizer'].get_feature_names_out()
coefficients = pipeline.named_steps['sgdclassifier'].coef_[0]
feature_importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
feature_importance['abs_coefficient'] = feature_importance['coefficient'].abs()
top_features = feature_importance.sort_values(by='abs_coefficient', ascending=False).head(20)
print(top_features[['feature', 'coefficient']])

               feature  coefficient
4136             image    13.728022
6646  president donald   -11.468980
2511             doesn    10.714105
7636              said   -10.304100
2367              didn     9.135440
5961           october     9.005516
9045      told reuters    -8.756525
9285            trumps    -8.657807
416                 ap     8.633501
6657   president trump     8.459098
9775               wfb     8.437923
5023                ll     7.818502
7719           saidthe     7.807142
4485               isn     7.774692
9305           tuesday    -7.614313
2533               don     7.231390
9693              wasn     7.176788
2909               est    -7.055250
5908               nyp     6.907614
9514                ve     6.738964
